**BRFSS Risk Factors for Heart Disease Data EDA**

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/brfss/brfss_survey_data_2024.csv")

# print first entries
df.head()

# number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 457670
Number of columns: 301


In [7]:
# print all column names
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

# print names of columns and stats on each column 
eda_table = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(dropna=False).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2)
})

eda_table = eda_table.sort_values("missing_percent", ascending=False)
eda_table.to_csv("eda_basics_table.csv", index=False)

1. _STATE
2. FMONTH
3. IDATE
4. IMONTH
5. IDAY
6. IYEAR
7. DISPCODE
8. SEQNO
9. _PSU
10. CTELENM1
11. PVTRESD1
12. COLGHOUS
13. STATERE1
14. CELPHON1
15. LADULT1
16. NUMADULT
17. RESPSLC1
18. LANDSEX3
19. SAFETIME
20. CTELNUM1
21. CELLFON5
22. CADULT1
23. CELLSEX3
24. PVTRESD3
25. CCLGHOUS
26. CSTATE1
27. LANDLINE
28. HHADULT
29. SEXVAR
30. GENHLTH
31. PHYSHLTH
32. MENTHLTH
33. POORHLTH
34. PRIMINS2
35. PERSDOC3
36. MEDCOST1
37. CHECKUP1
38. EXERANY2
39. LASTDEN4
40. RMVTETH4
41. CVDINFR4
42. CVDCRHD4
43. CVDSTRK3
44. ASTHMA3
45. ASTHNOW
46. CHCSCNC1
47. CHCOCNC1
48. CHCCOPD3
49. ADDEPEV3
50. CHCKDNY2
51. HAVARTH4
52. DIABETE4
53. DIABAGE4
54. MARITAL
55. EDUCA
56. RENTHOM1
57. NUMHHOL4
58. NUMPHON4
59. CPDEMO1C
60. VETERAN3
61. EMPLOY1
62. CHILDREN
63. INCOME3
64. PREGNANT
65. WEIGHT2
66. HEIGHT3
67. DEAF
68. BLIND
69. DECIDE
70. DIFFWALK
71. DIFFDRES
72. DIFFALON
73. HADMAM
74. HOWLONG
75. CERVSCRN
76. CRVCLCNC
77. CRVCLPAP
78. CRVCLHPV
79. HADHYST2
80. HADSIGM4
81. COLNSIGM
82. COLN

In [8]:
eda_numerical_summary = df.describe()
eda_numerical_summary.to_csv("eda_numerical_summary.csv", index=False)

Useful columns:   
**_MICHD** is "Respondents that have ever reported having coronary heart disease (CHD) or myocardial infarction (MI)": 1=Yes, 2=No    
**CVDINFR4** is "(Ever told) you had a heart attack, also called a myocardial infarction?"  
**CVDCRHD4** is "(Ever told) (you had) angina or coronary heart disease?"  


In [ ]:
# function that for each feature calulates the percentage of people in each category that have heart disease
def heart_disease_rate_by_feature(data, feature, target="_MICHD"):
    if feature not in data.columns:
        raise ValueError(f"{feature} is not in the dataframe columns.")
    
    if target not in data.columns:
        raise ValueError(f"{target} is not in the dataframe columns.")

    temp = data[[feature, target]].copy()
    temp = temp[temp[target].isin([1, 2])]
    temp = temp.dropna(subset=[feature])

    summary = (
        temp
        .groupby(feature)
        .agg(
            total_people=(target, "count"),
            heart_disease_cases=(target, lambda x: (x == 1).sum())
        )
        .reset_index()
    )

    summary["heart_disease_percent"] = (
        summary["heart_disease_cases"] / summary["total_people"] * 100
    ).round(2)

    summary = summary.rename(columns={feature: "feature_value"})

    summary.insert(0, "feature", feature)

    summary = summary.sort_values(
        "heart_disease_percent",
        ascending=False
    )

    return summary

**Variables that would be useful to include:**   
Demographics:  
_SEX : Sex  
_AGEG5YR : Age  
EDUCA : Education level
INCOME3 : Income 
EMPLOY1 : Employment status 
MARITAL : Marital status
RENTHOM1 : Own or rent home  

General Health:   
GENHLTH : "Would you say that in general your health is: good, ok, ..."
or
_RFHLTH : "Would you say that in general your health is excellent, very good, good, fair, or poor?"  
CHECKUP1 : "Time since last routine checkup"

Stats:  
HEIGHT3 : Height  
WEIGHT2 : Weight (Could use to calulate BMI)  

Lifestyle:  
SMOKE100 : "Have you smoked at least 100 cigarettes in your entire life?   [Note:  5 packs = 100 cigarettes]"  
SMOKDAY2 : “Do you now smoke cigarettes every day, some days, or not at all?”   
ECIGNOW3 : "Would you say you have never used e-cigarettes or other electronic vaping products in your entire life or now use them every day, use them some days, or used them in the past but do not currently use them at all?"    
USENOW3 :  "Current use of chewing tobacco, snuff, or snus"  
ALCDAY4 : "During the past 30 days, how many days per week or per month did you have at least one drink of any alcoholic beverage?  (A 40 ounce beer would count as 3 drinks, or a cocktail drink with 2 shots would count as 2 drinks.)"   

Medical History:  
DIABETE4 : “Ever told you had diabetes?”   
CHCKDNY2 : “Ever told you had kidney disease?”  
CVDSTRK3 : “Ever told you had a stroke?”  
CHCCOPD3 : “Ever told you had COPD, emphysema, or chronic bronchitis?”  
ADDEPEV3 : “Ever told you had a depressive disorder?”  
HAVARTH4 : “Ever told you had some form of arthritis?”  

In [21]:
predictors = [
    "_SEX",
    "_AGEG5YR",
    "EDUCA",
    "INCOME3",
    "EMPLOY1",
    "MARITAL",
    "RENTHOM1",
    "GENHLTH",
    "_RFHLTH",
    "CHECKUP1",
    "HEIGHT3",
    "WEIGHT2",
    "SMOKE100",
    "SMOKDAY2",
    "ECIGNOW3",
    "USENOW3",
    "ALCDAY4",
    "DIABETE4",
    "CHCKDNY2",
    "CVDSTRK3",
    "CHCCOPD3",
    "ADDEPEV3",
    "HAVARTH4"
]

heart_disease_dfs = {}

for predictor in predictors:
    heart_disease_dfs[predictor] = heart_disease_rate_by_feature(df, predictor)

for predictor, result_df in heart_disease_dfs.items():
    print(f"\nResults for {predictor}")
    display(result_df)


Results for _SEX


,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
0,_SEX,1.0,214790,24530,11.42
1,_SEX,2.0,237674,17808,7.49



Results for _AGEG5YR


,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
12,_AGEG5YR,13.0,40715,9336,22.93
11,_AGEG5YR,12.0,36185,6997,19.34
10,_AGEG5YR,11.0,44268,6925,15.64
9,_AGEG5YR,10.0,47220,6038,12.79
8,_AGEG5YR,9.0,42960,4575,10.65
7,_AGEG5YR,8.0,34604,2838,8.20
13,_AGEG5YR,14.0,8023,581,7.24
6,_AGEG5YR,7.0,31403,1894,6.03
5,_AGEG5YR,6.0,28757,1088,3.78
4,_AGEG5YR,5.0,30669,768,2.50



Results for EDUCA


...

,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
0,CHCCOPD3,1.0,35991,9860,27.40
2,CHCCOPD3,7.0,1693,324,19.14
3,CHCCOPD3,9.0,89,16,17.98
1,CHCCOPD3,2.0,414686,32136,7.75



Results for ADDEPEV3


,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
2,ADDEPEV3,7.0,1888,217,11.49
0,ADDEPEV3,1.0,94861,10827,11.41
3,ADDEPEV3,9.0,422,45,10.66
1,ADDEPEV3,2.0,355290,31249,8.80



Results for HAVARTH4


,feature,feature_value,total_people,heart_disease_cases,heart_disease_percent
0,HAVARTH4,1.0,155888,25630,16.44
2,HAVARTH4,7.0,2069,222,10.73
3,HAVARTH4,9.0,141,11,7.80
1,HAVARTH4,2.0,294363,16475,5.60


In [23]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

outcome = "_MICHD"

results = []

for predictor in predictors:
    temp = df[[predictor, outcome]].dropna()
    table = pd.crosstab(temp[predictor], temp[outcome])

    if table.shape[0] < 2 or table.shape[1] < 2:
        continue

    chi2, p_value, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    min_dim = min(table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

    results.append({
        "predictor": predictor,
        "chi2": chi2,
        "p_value": p_value,
        "statistically_significant": "yes" if p_value < 0.05 else "no",
        "cramers_v": cramers_v,
        "degrees_of_freedom": dof,
        "n": n
    })

significance_results_df = (
    pd.DataFrame(results)
    .sort_values(["p_value", "cramers_v"], ascending=[True, False])
    .reset_index(drop=True)
)

significance_results_df

,predictor,chi2,p_value,statistically_significant,cramers_v,degrees_of_freedom,n
0,_AGEG5YR,27606.608103,0.000000e+00,yes,0.247010,13,452464
1,GENHLTH,26206.111471,0.000000e+00,yes,0.240664,6,452461
2,EMPLOY1,23658.930506,0.000000e+00,yes,0.229493,8,449217
3,_RFHLTH,19814.332551,0.000000e+00,yes,0.209266,2,452464
4,CVDSTRK3,18348.354388,0.000000e+00,yes,0.201376,3,452463
5,CHCCOPD3,15270.544798,0.000000e+00,yes,0.183712,3,452459
6,DIABETE4,14755.853396,0.000000e+00,yes,0.180589,5,452462
7,HAVARTH4,14136.081915,0.000000e+00,yes,0.176756,3,452461
8,CHCKDNY2,11504.026612,0.000000e+00,yes,0.159454,3,452460
9,MARITAL,8890.847049,0.000000e+00,yes,0.140179,6,452457
